# Carga em Massa — analise_siplan_rps

Enriquece a tabela `wh_siplan_rps.dbo.analise_siplan_rps` com três novos campos vindos do lakehouse:

| Campo | Origem |
|-------|--------|
| `justificativa` | `lake_gold_fatos.dbo.base` |
| `todas_as_tags` | `lake_gold_fatos.dbo.base` |
| `info_pcap` | `lake_gold_fatos.dbo.pcap` (agregado por `atividade_id`) |

## Como funciona

1. Lê o estado atual de `analise_siplan_rps` via JDBC (preserva `Status`, `custos_foto`, e-mails, datas).
2. Lê `base` e `pcap` do lakehouse via Spark.
3. Faz LEFT JOIN e acrescenta / atualiza os três novos campos.
4. Regrava a tabela inteira com `mode=overwrite` + `batchsize=1000`.

> **Quando rodar**: poucas vezes (após atualização do lakehouse gold ou adição dos campos).  
> **Não substitui** `nb_fluxo_autorizacao` — esse continua registrando novos INSERTs/UPDATEs via PA.

In [ ]:
# ── Parâmetros ──────────────────────────────────────────────────────────────
SQL_ENDPOINT  = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)
WAREHOUSE_DB  = 'wh_siplan_rps'
TARGET_TABLE  = 'analise_siplan_rps'

BASE_TABLE    = 'lake_gold_fatos.dbo.base'
PCAP_TABLE    = 'lake_gold_fatos.dbo.pcap'

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from datetime import datetime
from pyspark.sql import functions as F, Window

try:
    from notebookutils import mssparkutils
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

print(f'Início: {datetime.now():%d/%m/%Y %H:%M}')

In [ ]:
# ── Token e JDBC URL (reutilizados nas leituras e na escrita) ─────────────────
def get_db_token() -> str:
    if HAS_MSSPARKUTILS:
        return mssparkutils.credentials.getToken('https://database.windows.net/.default')
    from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    return cred.get_token('https://database.windows.net/.default').token


jdbc_url = (
    f'jdbc:sqlserver://{SQL_ENDPOINT}'
    f';database={WAREHOUSE_DB}'
    f';encrypt=true;trustServerCertificate=false'
)

# ── Leitura do estado atual do warehouse ─────────────────────────────────────
print(f'Lendo {WAREHOUSE_DB}.dbo.{TARGET_TABLE}...')
df_analise = (
    spark.read
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', f'dbo.{TARGET_TABLE}')
    .option('accessToken', get_db_token())
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .load()
)

n_analise = df_analise.count()
print(f'{n_analise} registros lidos de {WAREHOUSE_DB}.dbo.{TARGET_TABLE}')

In [ ]:
# ── Leitura de justificativa e todas_as_tags da tabela base ──────────────────
print(f'Lendo {BASE_TABLE}...')
df_base = spark.sql(
    f'SELECT atividade_id, justificativa, todas_as_tags FROM {BASE_TABLE}'
)
n_base = df_base.count()
print(f'{n_base} registros em {BASE_TABLE}')

In [ ]:
# ── Leitura e agregação de pcap por atividade_id ──────────────────────────────
print(f'Lendo {PCAP_TABLE}...')
df_pcap_raw = spark.sql(
    'SELECT atividade_id, proponente, pcap_total, det_grupos, status '
    f'FROM {PCAP_TABLE}'
)

# Formata pcap_total como R$ inteiro — ponto como separador de milhar (pt-BR)
pcap_fmt = F.regexp_replace(
    F.format_number(F.col('pcap_total').cast('long'), 0),
    ',', '.'
)

# Bloco de texto por linha de PCAP (cada campo em linha separada)
bloco_pcap = F.concat_ws('\n',
    F.concat(F.lit('Proponente: '), F.coalesce(F.col('proponente'), F.lit(''))),
    F.concat(F.lit('Total: R$ '),   pcap_fmt),
    F.concat(F.lit('Status: '),     F.coalesce(F.col('status'),     F.lit(''))),
    F.concat(F.lit('Grupos: '),     F.coalesce(F.col('det_grupos'), F.lit(''))),
)

# Conta quantas PCAPs por atividade; numera o bloco somente quando há mais de uma
w_count = Window.partitionBy('atividade_id')
w_order = Window.partitionBy('atividade_id').orderBy('pcap_total')

df_pcap_agg = (
    df_pcap_raw
    .withColumn('n_pcap', F.count('*').over(w_count))
    .withColumn('n',      F.row_number().over(w_order))
    .withColumn('bloco',
        F.when(F.col('n_pcap') > 1,
               F.concat(F.lit('['), F.col('n'), F.lit(']\n'), bloco_pcap))
        .otherwise(bloco_pcap)
    )
    .groupBy('atividade_id')
    .agg(F.concat_ws('\n\n', F.collect_list('bloco')).alias('info_pcap'))
)

n_pcap = df_pcap_agg.count()
print(f'{n_pcap} atividades com PCAP')
df_pcap_agg.show(5, truncate=120)

In [ ]:
# ── Atividades que compartilham a mesma PCAP (self-join em pcap_num) ─────────
# Grain de pcap: atividade_id + pcap_num (sem duplicatas).
# Para cada atividade, mesma_pcap lista os demais atividade_ids que referenciam
# pelo menos um pcap_num em comum, ordenados e separados por vírgula.
# NULL quando a atividade não tem PCAP ou é a única a referenciá-la.

df_pcap_ids = spark.sql(
    f'SELECT DISTINCT atividade_id, pcap_num FROM {PCAP_TABLE}'
)

df_mesma = (
    df_pcap_ids.alias('a')
    .join(df_pcap_ids.alias('b'), on='pcap_num', how='inner')
    .filter(F.col('a.atividade_id') != F.col('b.atividade_id'))
    .select(
        F.col('a.atividade_id').alias('atividade_id'),
        F.col('b.atividade_id').cast('long').cast('string').alias('outra_id'),
    )
    .distinct()
)

df_mesma_pcap = (
    df_mesma
    .groupBy('atividade_id')
    .agg(
        F.concat_ws(', ', F.sort_array(F.collect_set('outra_id'))).alias('mesma_pcap')
    )
)

n_mesma = df_mesma_pcap.count()
print(f'{n_mesma} atividades com PCAP compartilhada')
df_mesma_pcap.show(5, truncate=80)

In [ ]:
# ── Join dos novos campos preservando todas as colunas existentes ─────────────
COLUNAS_NOVAS = {'justificativa', 'todas_as_tags', 'info_pcap', 'mesma_pcap'}

# Remove colunas novas se já existirem no df_analise (evita conflito no join)
colunas_drop = COLUNAS_NOVAS & set(df_analise.columns)
if colunas_drop:
    df_analise = df_analise.drop(*colunas_drop)
    print(f'Colunas removidas antes do join: {sorted(colunas_drop)}')

df_final = (
    df_analise
    .join(df_base,       on='atividade_id', how='left')
    .join(df_pcap_agg,   on='atividade_id', how='left')
    .join(df_mesma_pcap, on='atividade_id', how='left')
)

n_final = df_final.count()
print(f'df_final: {n_final} registros')
assert n_final == n_analise, f'Contagem mudou após o join: {n_analise} → {n_final}'
print('Colunas:', df_final.columns)

In [ ]:
# ── Escrita no warehouse: ALTER + TRUNCATE + APPEND ─────────────────────────
# Fabric Warehouse nao suporta os tipos que o Spark gera no CREATE TABLE
# (datetime, nvarchar(max)). Estrategia: a tabela deve existir (criada pelo
# nb_AnaliseSiplanRPS); adicionamos colunas novas se possivel, truncamos e
# fazemos APPEND somente com as colunas que existirem na tabela.

def _jdbc_exec(sql: str) -> None:
    token = mssparkutils.credentials.getToken('https://database.windows.net/')
    ds = spark._jvm.com.microsoft.sqlserver.jdbc.SQLServerDataSource()
    ds.setServerName(SQL_ENDPOINT)
    ds.setPortNumber(1433)
    ds.setDatabaseName(WAREHOUSE_DB)
    ds.setEncrypt(True)
    ds.setAccessToken(token)
    conn = ds.getConnection()
    stmt = conn.createStatement()
    stmt.execute(sql)
    conn.close()


# 1. Lê as colunas que existem hoje no warehouse (query vazia, sem dados)
colunas_warehouse = set(
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('query', f'SELECT TOP 0 * FROM dbo.{TARGET_TABLE}')
    .option('accessToken', get_db_token())
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .load()
    .columns
)
print(f'{len(colunas_warehouse)} colunas no warehouse.')

# 2. Tenta adicionar cada coluna nova com varios tipos; continua se falhar
# Se todos os tipos falharem, imprime instrucao manual e segue sem a coluna.
NOVAS_COLUNAS = ['justificativa', 'todas_as_tags', 'info_pcap', 'mesma_pcap']
TIPOS_TENTAR  = ['VARCHAR(MAX)', 'VARCHAR(4000)', 'NVARCHAR(4000)', 'VARCHAR(500)']

for col in NOVAS_COLUNAS:
    if col in colunas_warehouse:
        print(f'Coluna ja existe: {col}')
        continue
    adicionada = False
    for tipo in TIPOS_TENTAR:
        try:
            _jdbc_exec(f'ALTER TABLE dbo.{TARGET_TABLE} ADD [{col}] {tipo}')
            colunas_warehouse.add(col)
            print(f'Coluna adicionada: {col} ({tipo})')
            adicionada = True
            break
        except Exception as e:
            msg = str(e)
            if 'already' in msg.lower() or 'Column names in each table must be unique' in msg:
                colunas_warehouse.add(col)
                print(f'Coluna ja existe: {col}')
                adicionada = True
                break
            print(f'  {tipo}: {msg[:90]}')
    if not adicionada:
        print(f'AVISO: nao foi possivel adicionar [{col}] via JDBC.')
        print(f'  → Adicione manualmente no SQL Editor do Fabric:')
        print(f'    ALTER TABLE dbo.{TARGET_TABLE} ADD [{col}] VARCHAR(MAX)')

# 3. Seleciona apenas colunas que existem na tabela (append nao grava extras)
df_para_gravar = df_final.select(
    [c for c in df_final.columns if c in colunas_warehouse]
)
novas_presentes = [c for c in NOVAS_COLUNAS if c in df_para_gravar.columns]
print(f'Colunas novas incluidas na carga: {novas_presentes}')

# 4. Limita strings novas ao tamanho do campo
LIMITE_CHARS = {'justificativa': 4000, 'todas_as_tags': 4000,
                'info_pcap': 4000, 'mesma_pcap': 2000}
for col, n in LIMITE_CHARS.items():
    if col in df_para_gravar.columns:
        df_para_gravar = df_para_gravar.withColumn(col, F.substring(F.col(col), 1, n))

# 5. Materializa df_para_gravar ANTES do TRUNCATE
# Spark e lazy: sem cache, o pipeline releria o warehouse pos-truncate e gravaria 0 linhas.
df_para_gravar = df_para_gravar.cache()
n_gravar = df_para_gravar.count()  # forca a materializacao no cache Spark

if n_gravar == 0:
    df_para_gravar.unpersist()
    raise ValueError(
        'df_para_gravar esta vazio — abortando para nao truncar a tabela.\n'
        'Verifique se a tabela do warehouse tem dados e rode desde leitura-analise.'
    )
print(f'{n_gravar} registros materializados no cache Spark.')

_jdbc_exec(f'TRUNCATE TABLE dbo.{TARGET_TABLE}')
print(f'dbo.{TARGET_TABLE} truncada.')

print(f'Gravando {n_gravar} registros em {WAREHOUSE_DB}.dbo.{TARGET_TABLE}...')
print('(pode levar alguns minutos)')

# 6. Append — nao recria tabela, sem problema de tipo no CREATE TABLE
(
    df_para_gravar.write
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', f'dbo.{TARGET_TABLE}')
    .option('accessToken', get_db_token())
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .option('batchsize', 1000)
    .mode('append')
    .save()
)

print(f'OK — dbo.{TARGET_TABLE} atualizada em {WAREHOUSE_DB}.')

In [ ]:
# ── Verificação ───────────────────────────────────────────────────────────────
count = (
    spark.read
    .format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', f'dbo.{TARGET_TABLE}')
    .option('accessToken', get_db_token())
    .option('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    .load()
    .count()
)
print(f'Verificação: {count} registros em {WAREHOUSE_DB}.dbo.{TARGET_TABLE}')
print(f'Fim: {datetime.now():%d/%m/%Y %H:%M}')

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(str(count))